In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import glob
import os
import subprocess
import sys

sys.path.append("..")

from gatherData import roofline_results_to_df, calc_roofline_data

In [2]:
ncu_files = glob.glob(os.path.join("../../HeCBench/src", "atomicPerf-cuda", "*.ncu-rep"))

ncu_files = [os.path.abspath(f) for f in ncu_files]
print(ncu_files)



['/home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-3.ncu-rep', '/home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-1.ncu-rep', '/home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-0.ncu-rep', '/home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-noDisplay.ncu-rep', '/home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-2.ncu-rep', '/home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-docker-3.ncu-rep', '/home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-5.ncu-rep', '/home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-7.ncu-rep', '/home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-docker-1.ncu-rep', '/home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-4.ncu-rep', '/home/gbolet/gpu

In [3]:
def convert_ncu_to_csv(ncu_file):
    """
    Convert an NVIDIA Nsight Compute report file to a CSV file.
    """

    try:
        result = subprocess.run(
            [
                'ncu', '--import', ncu_file,
                '--csv', '--print-units', 'base', '--page', 'raw',
                '--print-kernel-base', 'mangled'
            ],
            timeout=60,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT
        )

        if result.returncode != 0:
            print(f'  Failed to parse ncu-rep file')
            return None

        print(f'  Successfully parsed ncu-rep file')
        return result
    except Exception as e:
        print(f'  Error parsing ncu-rep: {e}')
        return None

In [4]:
df = pd.DataFrame()

for ncu_file in ncu_files:
    print(f'Processing {ncu_file}...')
    result = convert_ncu_to_csv(ncu_file)
    if result is not None:
        rawDF = roofline_results_to_df(result)
        updatedDF = calc_roofline_data(rawDF)
        updatedDF['Source File'] = os.path.basename(ncu_file).replace('.ncu-rep', '')
        df = pd.concat([df, updatedDF], ignore_index=True)
        print(f'  Successfully converted {ncu_file}')
    else:
        print(f'  Failed to convert {ncu_file}')

Processing /home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-3.ncu-rep...
  Successfully parsed ncu-rep file
  Successfully converted /home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-3.ncu-rep
Processing /home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-1.ncu-rep...
  Successfully parsed ncu-rep file
  Successfully converted /home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-1.ncu-rep
Processing /home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-0.ncu-rep...
  Successfully parsed ncu-rep file
  Successfully converted /home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-0.ncu-rep
Processing /home/gbolet/gpuFLOPBench-updated/HeCBench/src/atomicPerf-cuda/atomicPerf-cuda-host-noDisplay.ncu-rep...
  Successfully parsed ncu-rep file
  Successfully converted /home/gbolet/gpuFLOPBench-updated/HeCB

In [5]:
print(df.head(3))

    ID  Process ID Process Name  Host Name  \
0  0.0      8451.0   atomicPerf  127.0.0.1   
1  1.0      8451.0   atomicPerf  127.0.0.1   
2  2.0      8451.0   atomicPerf  127.0.0.1   

                                 Kernel Name  Context  Stream   Block Size  \
0    _Z27BlockRangeAtomicOnGlobalMemIdEvPT_i      1.0     7.0  (256, 1, 1)   
1     _Z26WarpRangeAtomicOnGlobalMemIdEvPT_i      1.0     7.0  (256, 1, 1)   
2  _Z28SingleRangeAtomicOnGlobalMemIdEvPT_ii      1.0     7.0  (256, 1, 1)   

      Grid Size  Device  ...  hpAI      xtime                   device  \
0  (6048, 1, 1)     0.0  ...   0.0   286208.0  NVIDIA GeForce RTX 3080   
1  (6048, 1, 1)     0.0  ...   0.0  1136896.0  NVIDIA GeForce RTX 3080   
2  (6048, 1, 1)     0.0  ...   0.0  3405408.0  NVIDIA GeForce RTX 3080   

  SP_FLOP DP_FLOP HP_FLOP       INTOP       intPerf        intAI  \
0       0       0       0   7741440.0  2.704830e+10  1061.052632   
1       0       0       0  12386304.0  1.089484e+10  2481.230769   
2

In [6]:
# get only the kernel we are interested in
kname = '_Z28SingleRangeAtomicOnSharedMemIdEvPT_ii' 

print('mangled kernel name:', kname)
print('unmangled kernel name:', subprocess.check_output(['cu++filt', kname]).decode().strip())

df = df[df['Kernel Name'] == kname]

df['execEnv'] = df['Source File'].apply(lambda x: 'docker' if 'docker' in x else 'host')

print(df.shape)

mangled kernel name: _Z28SingleRangeAtomicOnSharedMemIdEvPT_ii
unmangled kernel name: void SingleRangeAtomicOnSharedMem<double>(T1 *, int, int)
(14, 977)


/tmp/ipykernel_4817/1974646328.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['execEnv'] = df['Source File'].apply(lambda x: 'docker' if 'docker' in x else 'host')


In [7]:
# let's print the SASS, it's short

!cat /home/gbolet/gpuFLOPBench-updated/atomicPerf-cuda-host-SingleRangeAtomicOnSharedMem.sass



Fatbin elf code:
arch = sm_86
code version = [1,8]
host = linux
compile_size = 64bit

	code for sm_86
		Function : _Z28SingleRangeAtomicOnSharedMemIdEvPT_ii
	.headerflags	@"EF_CUDA_64BIT_ADDRESS EF_CUDA_SM86 EF_CUDA_VIRTUAL_SM(EF_CUDA_SM86)"
	.headerflags	@"EF_CUDA_64BIT_ADDRESS EF_CUDA_SM86 EF_CUDA_VIRTUAL_SM(EF_CUDA_SM86)"
        /*0000*/                   MOV R1, c[0x0][0x28] ;                            /* 0x00000a0000017a02 */
                                                                                     /* 0x000fe40000000f00 */
        /*0010*/                   S2R R2, SR_CTAID.X ;                              /* 0x0000000000027919 */
                                                                                     /* 0x000e220000002500 */
        /*0020*/                   BSSY B0, 0x160 ;                                  /* 0x0000013000007945 */
                                                                                     /* 0x000fe60003800000 */
        /*00

In [11]:
# what we can clearly note below is that for 'some' runs, this particular kernel
# will occasional read huge amounts of data from DRAM, dragging it's arithmetic intensity down
# it's not entirely clear why this happens.
print(df[['execEnv', 'bytesRead', 'bytesWrite', 'bytesTotal', 'DP_FLOP', 'dpAI', 'Source File']].sort_values(by=['execEnv', 'bytesTotal'], ascending=False).reset_index(drop=True))

   execEnv  bytesRead  bytesWrite  bytesTotal    DP_FLOP          dpAI  \
0     host      29568         768       30336  117403142   3870.093057   
1     host       4864        1024        5888  117252339  19913.780526   
2     host       4224         384        4608  117950033  25596.795431   
3     host       4480           0        4480  117415109  26208.729774   
4     host       3200         256        3456  118102413  34173.151965   
5     host       3072           0        3072  117052849  38103.141114   
6     host       2944         128        3072  117427318  38225.038518   
7     host       2944           0        2944  117682139  39973.552771   
8     host       2944           0        2944  117846503  40029.382891   
9     host       2944           0        2944  117652822  39963.594612   
10  docker   18186624         512    18187136  117723037      6.472874   
11  docker       2944         128        3072  117759794  38333.266309   
12  docker       3072           0     

In [9]:
# we already diffed the SASS code of the two, and it came up identical
print(df['Block Size'].value_counts())
print(df['Grid Size'].value_counts())

Block Size
(256, 1, 1)    14
Name: count, dtype: int64
Grid Size
(6048, 1, 1)    14
Name: count, dtype: int64


In [10]:
!nvidia-smi

Sat Jan 24 21:48:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3080        On  |   00000000:01:00.0 Off |                  N/A |
|  0%   36C    P8              7W /  340W |       1MiB /  10240MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Solution:

It turns out that what we were seeing was a result of the `Disp.A` being on, causing 21 MiB of the GPU to always be in use for video buffer output. This explains why we saw that peak of 18MiB in one of the runs.
The DRAM counters offered by NVIDIA measure ALL DRAM traffic, including display output buffer reads/writes. This was inflating our DRAM R/W values. 

If we look at the CUDA kernel, we can notice that although it has a conditional to write to memory, that conditional never triggers. Thus, the DRAM writes should be 0. It also does everything in shared memory, so for it's computation, it never even writes/reads bytes from DRAM either. 

However, there is a catch. The DRAM counters can show reads/writes due to:
- Instruction Fetch + constant/parameter reads: These usually hit in constant/L1/L2 caches, but on cold start (or eviction) they can miss and ultimately come from DRAM. First invocation often pulls code/constant lines into caches which can show up as a few KB of DRAM traffic even though the kernel is “on-chip.”